# Wilcoxon Signed-Rank Tests for Anomaly Detection Benchmark

This notebook runs paired Wilcoxon signed-rank tests on the benchmark results to evaluate:
1. **Augmentation Gains** — Does augmented training improve AUROC over clean training under degradation?
2. **Overall Rescue Deltas** — Do rescue methods recover AUROC compared to degraded baselines?
3. **Per-Rescue-Method Breakdown** — Which specific rescue methods help or hurt, with statistical significance?

## Setup & Dependencies

In [ ]:
!pip install -q pandas scipy matplotlib seaborn statsmodels


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Match plotting style from generate_figures.py
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.titlesize': 14,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'figure.dpi': 150
})

os.makedirs('results/analysis', exist_ok=True)

## Load Dataset

Auto-detects your environment:
- **Google Colab**: Prompts you to upload `benchmark_full_4way.csv`
- **Kaggle**: Looks for the file in `/kaggle/input/`
- **Local**: Looks in the current working directory

In [ ]:
CSV_FILENAME = 'benchmark_full_4way.csv'

def load_csv():
    """Auto-detect environment and load the benchmark CSV."""
    # 1. Current directory
    if os.path.exists(CSV_FILENAME):
        print(f'Found {CSV_FILENAME} in current directory.')
        return pd.read_csv(CSV_FILENAME)

    # 2. Kaggle
    kaggle_base = '/kaggle/input'
    if os.path.isdir(kaggle_base):
        for root, dirs, files in os.walk(kaggle_base):
            if CSV_FILENAME in files:
                path = os.path.join(root, CSV_FILENAME)
                print(f'Found {CSV_FILENAME} at {path}')
                return pd.read_csv(path)
        print(f'CSV not found under {kaggle_base}. '
              'Make sure you have added the dataset to your Kaggle notebook.')

    # 3. Google Colab
    try:
        from google.colab import files as colab_files
        print(f"Please upload '{CSV_FILENAME}':")
        uploaded = colab_files.upload()
        if CSV_FILENAME in uploaded:
            return pd.read_csv(CSV_FILENAME)
        else:
            raise FileNotFoundError(
                f"Expected '{CSV_FILENAME}' but received: {list(uploaded.keys())}")
    except ImportError:
        pass

    raise FileNotFoundError(
        f"Could not find '{CSV_FILENAME}'. "
        'Place it in the current directory, upload it (Colab), '
        'or add it as a Kaggle dataset.')

# --- Call at top level (NOT inside load_csv) ---
df = load_csv()
print(f'Loaded successfully. Shape: {df.shape}')
df.head()

---
## Test 1 — Augmentation Gains

Compares **clean-trained** vs **augmented-trained** models under degradation (no rescue).  
Paired by `(model, category, seed, ctype, severity)`.  
One-sided test: H₁ — augmented training yields higher AUROC.

In [ ]:
# Filter to degradation phase with no rescue
deg_df = df[(df['phase'] == 'degradation') & (df['rescue'] == 'none')]

clean_deg = deg_df[deg_df['training'] == 'clean']
aug_deg   = deg_df[deg_df['training'] == 'augmented']

merge_cols = ['model', 'category', 'seed', 'ctype', 'severity']
aug_gains_df = pd.merge(
    clean_deg[merge_cols + ['image_AUROC']],
    aug_deg[merge_cols + ['image_AUROC']],
    on=merge_cols,
    suffixes=('_clean_train', '_aug_train')
)

print('=' * 60)
print('1. Wilcoxon Signed-Rank Test: Augmentation Gains')
print('=' * 60)

test1_results = []

for model in ['PatchCore', 'PaDiM']:
    model_df = aug_gains_df[aug_gains_df['model'] == model]
    if len(model_df) == 0:
        continue

    x = model_df['image_AUROC_clean_train']
    y = model_df['image_AUROC_aug_train']

    # One-sided: H1: clean < augmented
    stat, pval = wilcoxon(x, y, alternative='less')
    mean_diff = (y - x).mean()

    test1_results.append({
        'Model': model, 'N': len(model_df),
        'Mean Clean': x.mean(), 'Mean Aug': y.mean(),
        'Mean Gain': mean_diff, 'W': stat, 'p-value': pval,
    })

    print(f'Model: {model} (N = {len(model_df)})')
    print(f'  Mean Clean-Train AUROC: {x.mean():.4f}')
    print(f'  Mean Aug-Train AUROC:   {y.mean():.4f}')
    print(f'  Mean Gain:              {mean_diff:+.4f}')
    print(f'  W-Statistic:            {stat}')
    print(f'  p-value:                {pval:.2e}')
    sig = 'Yes' if pval < 0.05 else 'No'
    print(f'  Significant (a=0.05)?   {sig}')
    print('-' * 50)

test1_df = pd.DataFrame(test1_results)

### Figure 7 — Augmentation Gains with Significance

In [ ]:
# Baseline clean AUROC reference values from benchmark
CLEAN_BASELINE = {'PatchCore': 0.9822, 'PaDiM': 0.9226}
AUG_BASELINE   = {'PatchCore': 0.9773, 'PaDiM': 0.8979}
MEAN_DEG_CLEAN = {'PatchCore': 0.7356, 'PaDiM': 0.6390}
MEAN_DEG_AUG   = {'PatchCore': 0.8588, 'PaDiM': 0.7400}

fig, ax = plt.subplots(figsize=(7, 5))

model_colors = {'PatchCore': '#2b5c8f', 'PaDiM': '#d95f02'}
bars = ax.bar(
    test1_df['Model'],
    test1_df['Mean Gain'] * 100,
    color=[model_colors[m] for m in test1_df['Model']],
    edgecolor='black', linewidth=0.5, width=0.5
)

for bar, (_, row) in zip(bars, test1_df.iterrows()):
    h = bar.get_height()
    p = row['p-value']
    stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    ax.text(bar.get_x() + bar.get_width() / 2., h + 0.3,
            f'{h:+.1f} pp\n({stars})\nN={int(row["N"])}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Mean AUROC Gain (percentage points)')

# Secondary x-axis table: show absolute degraded AUROC for context
for i, (_, row) in enumerate(test1_df.iterrows()):
    m = row['Model']
    ax.text(i, -1.8, f'Clean deg: {MEAN_DEG_CLEAN[m]:.3f}', ha='center',
            fontsize=7.5, color='#555555')
    ax.text(i, -2.8, f'Aug deg:   {MEAN_DEG_AUG[m]:.3f}', ha='center',
            fontsize=7.5, color='#555555')
    ax.text(i, -3.8, f'(baseline: {CLEAN_BASELINE[m]:.3f})', ha='center',
            fontsize=7, color='#888888', style='italic')
ax.set_ylim(-5, max(test1_df['Mean Gain'] * 100) + 4)
ax.set_title('Robustness Gain from Augmented Training\n(Wilcoxon one-sided; * p<0.05, ** p<0.01, *** p<0.001)')
ax.grid(axis='y')

plt.tight_layout()
fig.savefig('results/analysis/07_wilcoxon_augmentation_gains.png', bbox_inches='tight')
fig.savefig('07_wilcoxon_augmentation_gains.png', bbox_inches='tight')
plt.show()

---
## Test 2 — Overall Rescue Deltas (Rescued vs. Degraded)

Compares **rescued AUROC** vs **degraded AUROC**.  
Paired by `(model, training, category, seed, ctype, severity)`.  
Two-sided test for any difference + one-sided test for rescue helping (degraded < rescued).

In [ ]:
rescue_phase_df = df[df['phase'] == 'rescue']
rescue_pairs = pd.merge(
    rescue_phase_df,
    deg_df,
    on=['model', 'training', 'category', 'seed', 'ctype', 'severity'],
    suffixes=('_rescue', '_deg')
)

print('=' * 60)
print('2. Wilcoxon Signed-Rank Test: Overall Rescue Deltas')
print('=' * 60)

test2_results = []

for (model, training), group in rescue_pairs.groupby(['model', 'training']):
    x = group['image_AUROC_deg']
    y = group['image_AUROC_rescue']

    stat_two, pval_two = wilcoxon(x, y, alternative='two-sided')
    stat_one, pval_one = wilcoxon(x, y, alternative='less')

    mean_diff    = (y - x).mean()
    success_rate = (y > x).mean() * 100

    label = f'{model} ({training})'
    test2_results.append({
        'Condition': label, 'Model': model, 'Training': training,
        'N': len(group), 'Mean Delta': mean_diff,
        'Success Rate': success_rate,
        'p_two': pval_two, 'p_one': pval_one,
    })

    print(f'Condition: {model} ({training} training) (N = {len(group)})')
    print(f'  Mean Degraded AUROC:      {x.mean():.4f}')
    print(f'  Mean Rescued AUROC:       {y.mean():.4f}')
    print(f'  Mean Delta (Rescue):      {mean_diff:+.4f}')
    print(f'  Rescue Success Rate:      {success_rate:.1f}%')
    print(f'  Two-sided W-Statistic:    {stat_two}')
    print(f'  Two-sided p-value:        {pval_two:.2e}')
    print(f'  Rescued > Degraded p-val: {pval_one:.2e}')
    hurts = 'Yes' if (pval_two < 0.05 and mean_diff < 0) else 'No'
    print(f'  Rescue hurts significantly? {hurts}')
    print('-' * 50)

test2_df = pd.DataFrame(test2_results)

### Figure 8 — Rescue Deltas with Significance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Reorder: group by model so augmented/clean are side-by-side per model
model_order = ['PatchCore (clean)', 'PatchCore (augmented)', 'PaDiM (clean)', 'PaDiM (augmented)']
test2_df_ordered = test2_df.set_index('Condition').reindex(
    [m for m in model_order if m in test2_df['Condition'].values]).reset_index()

cond_colors = {'PatchCore (clean)': '#2b5c8f', 'PatchCore (augmented)': '#4daf4a',
               'PaDiM (clean)': '#d95f02', 'PaDiM (augmented)': '#984ea3'}

bars = ax.bar(
    test2_df_ordered['Condition'],
    test2_df_ordered['Mean Delta'] * 100,
    color=[cond_colors.get(c, '#888888') for c in test2_df_ordered['Condition']],
    edgecolor='black', linewidth=0.5, width=0.55
)
test2_df = test2_df_ordered  # use reordered for annotations

for bar, (_, row) in zip(bars, test2_df.iterrows()):
    h = bar.get_height()
    p = row['p_two']
    stars = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
    offset = -0.5 if h < 0 else 0.5
    va = 'top' if h < 0 else 'bottom'
    ax.text(bar.get_x() + bar.get_width() / 2., h + offset,
            f'{h:+.1f} pp\n({stars})', ha='center', va=va, fontweight='bold', fontsize=9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylim(-22, 4)

# Add a legend explaining the baseline context
baseline_rows = {
    'PatchCore (clean)':     {'clean_deg': 0.7356, 'clean_base': 0.9822},
    'PatchCore (augmented)': {'clean_deg': 0.8588, 'clean_base': 0.9773},
    'PaDiM (clean)':         {'clean_deg': 0.6390, 'clean_base': 0.9226},
    'PaDiM (augmented)':     {'clean_deg': 0.7400, 'clean_base': 0.8979},
}
for idx, cond in enumerate(test2_df['Condition']):
    if cond in baseline_rows:
        info = baseline_rows[cond]
        ax.text(idx, -20.5,
                f"Base: {info['clean_base']:.3f}\nDeg: {info['clean_deg']:.3f}",
                ha='center', va='bottom', fontsize=7, color='#444444',
                bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec='gray', alpha=0.8))
ax.set_ylabel('Mean Rescue Delta (percentage points)')
ax.set_ylim(-22, 4)
ax.set_title('Overall Pre-processing Rescue Effect on AUROC\n(Negative = harmful; * p<0.05, ** p<0.01, *** p<0.001)')
ax.grid(axis='y')
ax.set_xlabel('', labelpad=2)
fig.text(0.5, -0.04,
    'Note: aggregate across all 6 rescue methods; Wiener alone accounts for the majority of the negative effect (see Fig. 9).',
    ha='center', va='top', fontsize=7.5, style='italic', wrap=True)

plt.tight_layout()
fig.savefig('results/analysis/08_wilcoxon_rescue_deltas.png', bbox_inches='tight')
fig.savefig('08_wilcoxon_rescue_deltas.png', bbox_inches='tight')
plt.show()

---
## Test 3 — Per-Rescue-Method Wilcoxon Breakdown

Tests each rescue method individually per model & training condition.  
This provides statistical backing for claims like *"Wiener deconvolution is categorically harmful"* and *"CLAHE is the only conditionally beneficial rescue method"*.

In [ ]:
print('=' * 60)
print('3. Wilcoxon Signed-Rank Test: Per-Rescue-Method Breakdown')
print('=' * 60)

test3_results = []

for (model, training, rescue_method), group in rescue_pairs.groupby(
        ['model', 'training', 'rescue_rescue']):
    if len(group) < 10:
        continue

    x = group['image_AUROC_deg']
    y = group['image_AUROC_rescue']

    stat_two, pval_two = wilcoxon(x, y, alternative='two-sided')
    stat_one, pval_one = wilcoxon(x, y, alternative='less')

    mean_diff = (y - x).mean()
    success_rate = (y > x).mean() * 100

    test3_results.append({
        'Model': model, 'Training': training,
        'Rescue Method': rescue_method, 'N': len(group),
        'Mean Deg': x.mean(), 'Mean Res': y.mean(),
        'Mean Delta': mean_diff, 'Success %': success_rate,
        'p_two': pval_two, 'p_one': pval_one,
    })

test3_df = pd.DataFrame(test3_results)

# ── Apply Benjamini-Hochberg FDR correction across all 24 tests ──
_, pvals_fdr, _, _ = multipletests(test3_df['p_two'], method='fdr_bh')
test3_df['p_fdr'] = pvals_fdr
print(f"FDR correction applied. {(pvals_fdr < 0.05).sum()} / {len(pvals_fdr)} tests remain significant at q<0.05")

# Pretty-print per-method results
for (model, training), subdf in test3_df.groupby(['Model', 'Training']):
    print(f'\n--- {model} ({training} training) ---')
    for _, row in subdf.sort_values('Mean Delta').iterrows():
        p = row['p_two']
        p_fdr = row['p_fdr']
        stars_raw = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'n.s.'))
        stars_fdr = '***' if p_fdr < 0.001 else ('**' if p_fdr < 0.01 else ('*' if p_fdr < 0.05 else 'n.s.'))
        print(f"  {row['Rescue Method']:25s}  N={row['N']:3d}  "
              f"delta={row['Mean Delta']:+.4f}  success={row['Success %']:5.1f}%  "
              f"p={p:.2e} {stars_raw}  p_fdr={p_fdr:.2e} {stars_fdr}")

### Figure 9 — Per-Method Significance Heatmap

In [ ]:
# Build condition label for columns
test3_df['Condition'] = test3_df['Model'] + '\n(' + test3_df['Training'] + ')'

# Pivot delta values, raw p-values, and FDR-corrected p-values
pivot_delta = test3_df.pivot_table(
    index='Rescue Method', columns='Condition', values='Mean Delta')
pivot_p = test3_df.pivot_table(
    index='Rescue Method', columns='Condition', values='p_two')
pivot_p_fdr = test3_df.pivot_table(
    index='Rescue Method', columns='Condition', values='p_fdr')

# Reorder rows logically by corruption type
method_order = ['CLAHE', 'Retinex', 'Wiener', 'Wiener (Motion PSF)',
                'NLM Denoise', 'Dehaze (Dark Channel)']
pivot_delta = pivot_delta.reindex([m for m in method_order if m in pivot_delta.index])
pivot_p = pivot_p.reindex(pivot_delta.index)
pivot_p_fdr = pivot_p_fdr.reindex(pivot_delta.index)

# Build annotation strings with delta + significance stars
annot = np.empty(pivot_delta.shape, dtype=object)
for r in range(pivot_delta.shape[0]):
    for c in range(pivot_delta.shape[1]):
        v = pivot_delta.iloc[r, c]
        p = pivot_p.iloc[r, c]
        if np.isnan(v):
            annot[r, c] = ''
            continue
        # Use FDR-corrected p-value; always show significance label (n.s. if not significant)
        p_fdr = pivot_p_fdr.iloc[r, c]
        stars = '***' if p_fdr < 0.001 else ('**' if p_fdr < 0.01 else ('*' if p_fdr < 0.05 else 'n.s.'))
        annot[r, c] = f'{v*100:+.1f}pp\n{stars}'

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    pivot_delta * 100, cmap='RdBu', center=0, vmin=-35, vmax=15,
    annot=annot, fmt='', annot_kws={'fontsize': 9, 'weight': 'bold'},
    cbar_kws={'label': 'Mean Delta (AUROC pp)'},
    linewidths=0.5, linecolor='gray', ax=ax
)

ax.set_title('Per-Method Rescue Effect with Wilcoxon Significance\n'
             '(FDR-corrected; * q<0.05, ** q<0.01, *** q<0.001, n.s. = not significant)',
             pad=15, fontweight='bold')
ax.set_xlabel('Model & Training Regime')

# Annotate column headers with mean degraded AUROC reference
col_baselines = {
    'PaDiM\n(augmented)':   ('Base: 0.9226', 'Deg: 0.7400'),
    'PaDiM\n(clean)':       ('Base: 0.9226', 'Deg: 0.6390'),
    'PatchCore\n(augmented)': ('Base: 0.9822', 'Deg: 0.8588'),
    'PatchCore\n(clean)':   ('Base: 0.9822', 'Deg: 0.7356'),
}
for tick_idx, label in enumerate(pivot_delta.columns):
    if label in col_baselines:
        b, d = col_baselines[label]
        ax.text(tick_idx + 0.5, -0.7, f'{b} | {d}',
                ha='center', va='top', fontsize=7, color='#555555',
                transform=ax.get_xaxis_transform())
ax.set_ylabel('Rescue Method')
plt.xticks(rotation=0)

plt.tight_layout()
fig.savefig('results/analysis/09_wilcoxon_per_method_heatmap.png', bbox_inches='tight')
fig.savefig('09_wilcoxon_per_method_heatmap.png', bbox_inches='tight')
plt.show()

---
## Summary

| Test | What it measures | Key result |
|---|---|---|
| **Test 1** | Augmented vs Clean training under degradation | Augmented training significantly improves AUROC |
| **Test 2** | Overall rescue effect across all methods | Rescue is net-harmful in all 4 conditions |
| **Test 3** | Per-method rescue breakdown | Wiener is categorically harmful; CLAHE is the only conditionally neutral/positive method |

**Figures saved to `results/analysis/`:**
- `07_wilcoxon_augmentation_gains.png`
- `08_wilcoxon_rescue_deltas.png`
- `09_wilcoxon_per_method_heatmap.png`